In [8]:
import os
from pathlib import Path
import time
import json
import shutil
import pickle
import traceback
import argparse
import warnings
import boto3
import pprint


In [3]:
pp = pprint.PrettyPrinter(width=41, compact=True, indent=4)
warnings.filterwarnings('ignore')

logs = []

In [12]:
def prepare_directories(root):
    try:
        # input
        input_dir = os.path.join(root, 'input')
        os.makedirs(os.path.join(input_dir, 'profile'), exist_ok=True)
        os.makedirs(os.path.join(input_dir, 'data'), exist_ok=True) # input_data_ref.yml
        os.makedirs(os.path.join(input_dir, 'metadata'), exist_ok=True) # run_manifest.yml
        os.makedirs(os.path.join(input_dir, 'config'), exist_ok=True) # config.yml
        return input_dir
    except Exception as e:
        pp.pprint(e)
        logs.append(str(e))
        pass

In [13]:
root = str(Path().resolve())

In [14]:
prepare_directories(root)

'/home/ec2-user/SageMaker/gs-ds-env/tests/titanic/notebooks/sean/run_pm/input'

# s3 예제 위치
```
# 데이터셋
s3://gs-retail-awesome-data-ap-northeast-2/
  env=dev/
    project=gs25-sales-forecast/
      version=v1.0/
        train.parquet
        validation.parquet
        test.parquet
        metadata.yml

# 결과물
s3://gs-retail-awesome-model-ap-northeast-2/
  env=dev/
    project=gs25-sales-forecast/
      experiment=baseline-v1/
        model=store-daily-sales-forecast/
          algo=lightgbm/
            run_date=2026-02-28/
              run_id=20260228T143022Z_a7f3c8d1/
                model.tar.gz
                metrics.json
```
## 정리
```
┌─────────────────────────────────────────────────────────────────────┐
│  gs-retail-awesome-data-ap-northeast-2 (Input)                      │
│  └── env / project / version                                        │
│       └── train.parquet, validation.parquet, test.parquet           │
└─────────────────────────────────────────────────────────────────────┘
                              ↓  SageMaker Training
┌─────────────────────────────────────────────────────────────────────┐
│  gs-retail-awesome-model-ap-northeast-2 (Output)                    │
│  └── env / project / experiment / model / algo / run_date / run_id  │
│       └── model.tar.gz, metrics.json, artifacts/                    │
└─────────────────────────────────────────────────────────────────────┘
```

In [25]:
"""
prepare_datasets.ipynb - 프로젝트 데이터셋 S3 업로드
"""

# %% [markdown]
# # 프로젝트 데이터셋 준비 및 S3 업로드
# 
# 프로젝트별 고정 데이터셋을 버전 관리하여 S3에 업로드합니다.
# 
# ## S3 구조
# ```
# s3://gs-retail-awesome-data/
#   env={env}/
#     project={project}/
#       version={version}/
#         train.parquet
#         validation.parquet
#         test.parquet
#         metadata.yml
# ```

# %% Imports
import os
import boto3
import yaml
from datetime import datetime
from pathlib import Path
import tempfile

# %% Configuration
class DatasetConfig:
    """데이터셋 S3 설정"""
    
    REGION = "ap-northeast-2"
    BUCKET = f"gs-retail-awesome-data-{REGION}"
    ENV = "dev"
    PROJECT = "gs25-sales-forecast"
    VERSION = "v1.0"
    
    # 파일 포맷 (parquet, csv 등)
    FILE_FORMAT = "parquet"
    
    @classmethod
    def get_s3_prefix(cls, env: str = None, project: str = None, version: str = None) -> str:
        """S3 prefix 경로 생성"""
        return (
            f"env={env or cls.ENV}/"
            f"project={project or cls.PROJECT}/"
            f"version={version or cls.VERSION}"
        )
    
    @classmethod
    def get_s3_uri(cls, env: str = None, project: str = None, version: str = None) -> str:
        """전체 S3 URI 반환"""
        prefix = cls.get_s3_prefix(env, project, version)
        return f"s3://{cls.BUCKET}/{prefix}/"


# %% Dataset Preparation Functions
def create_metadata(output_dir: str, config: dict = None) -> str:
    """
    metadata.yml 생성 - 데이터셋 메타정보
    """
    default_config = {
        'version': DatasetConfig.VERSION,
        'created_at': datetime.utcnow().isoformat(),
        'created_by': 'ax-squad',
        'project': DatasetConfig.PROJECT,
        'environment': DatasetConfig.ENV,
        'description': 'GS25 매장별 일일 매출 예측 데이터셋',
        'files': {
            'train': {
                'filename': f'train.{DatasetConfig.FILE_FORMAT}',
                'description': '학습 데이터',
                'date_range': '2024-01-01 ~ 2025-12-31',
            },
            'validation': {
                'filename': f'validation.{DatasetConfig.FILE_FORMAT}',
                'description': '검증 데이터',
                'date_range': '2026-01-01 ~ 2026-01-31',
            },
            'test': {
                'filename': f'test.{DatasetConfig.FILE_FORMAT}',
                'description': '테스트 데이터',
                'date_range': '2026-02-01 ~ 2026-02-28',
            }
        },
        'schema': {
            'target_column': 'daily_sales',
            'feature_columns': [
                'store_id', 'date', 'day_of_week', 'is_holiday',
                'temperature', 'precipitation', 'promotion_flag'
            ],
        }
    }
    
    final_config = config if config else default_config
    filepath = os.path.join(output_dir, 'metadata.yml')
    
    with open(filepath, 'w', encoding='utf-8') as f:
        yaml.dump(final_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"✓ Created: {filepath}")
    return filepath


def create_placeholder_datasets(output_dir: str, file_format: str = "parquet") -> list:
    """
    placeholder 데이터셋 파일 생성 (touch)
    
    실제 사용시에는 이 함수 대신 실제 데이터를 복사하거나 생성하세요.
    """
    files = ['train', 'validation', 'test']
    created_files = []
    
    for name in files:
        filepath = os.path.join(output_dir, f'{name}.{file_format}')
        Path(filepath).touch()
        created_files.append(filepath)
        print(f"✓ Created (placeholder): {filepath}")
    
    return created_files


# %% S3 Upload Functions
def upload_file_to_s3(local_path: str, bucket: str, s3_key: str, dry_run: bool = False) -> str:
    """단일 파일 S3 업로드"""
    s3_uri = f"s3://{bucket}/{s3_key}"
    
    if dry_run:
        print(f"[DRY RUN] {local_path} -> {s3_uri}")
    else:
        s3_client = boto3.client('s3')
        s3_client.upload_file(local_path, bucket, s3_key)
        print(f"✓ Uploaded: {s3_uri}")
    
    return s3_uri


def upload_directory_to_s3(local_dir: str, bucket: str, s3_prefix: str, dry_run: bool = False) -> list:
    """디렉토리 전체 S3 업로드"""
    uploaded = []
    
    for filename in os.listdir(local_dir):
        local_path = os.path.join(local_dir, filename)
        if os.path.isfile(local_path):
            s3_key = f"{s3_prefix}/{filename}"
            s3_uri = upload_file_to_s3(local_path, bucket, s3_key, dry_run)
            uploaded.append(s3_uri)
    
    return uploaded


# %% Main Functions
def prepare_and_upload_placeholder(
    env: str = None,
    project: str = None,
    version: str = None,
    dry_run: bool = False
) -> dict:
    """
    Placeholder 데이터셋 생성 및 S3 업로드 (테스트용)
    """
    s3_prefix = DatasetConfig.get_s3_prefix(env, project, version)
    s3_uri = DatasetConfig.get_s3_uri(env, project, version)
    
    print("=" * 60)
    print("데이터셋 S3 업로드 (Placeholder)")
    print("=" * 60)
    print(f"S3 URI: {s3_uri}")
    print("=" * 60)
    
    with tempfile.TemporaryDirectory() as tmp_dir:
        print("\n[Step 1] Placeholder 파일 생성")
        create_placeholder_datasets(tmp_dir, DatasetConfig.FILE_FORMAT)
        create_metadata(tmp_dir)
        
        print("\n[Step 2] S3 업로드")
        uploaded = upload_directory_to_s3(
            tmp_dir, DatasetConfig.BUCKET, s3_prefix, dry_run
        )
    
    print("\n" + "=" * 60)
    print(f"완료! S3 Path: {s3_uri}")
    print("=" * 60)
    
    return {'s3_uri': s3_uri, 'uploaded_files': uploaded, 'dry_run': dry_run}


def upload_existing_datasets(
    train_path: str,
    validation_path: str,
    test_path: str,
    env: str = None,
    project: str = None,
    version: str = None,
    include_metadata: bool = True,
    metadata_config: dict = None,
    dry_run: bool = False
) -> dict:
    """
    기존 데이터셋 파일들을 S3에 업로드
    
    Args:
        train_path: 학습 데이터 로컬 경로
        validation_path: 검증 데이터 로컬 경로
        test_path: 테스트 데이터 로컬 경로
        env, project, version: S3 경로 구성요소
        include_metadata: metadata.yml 생성 여부
        metadata_config: 커스텀 메타데이터
        dry_run: True면 업로드 시뮬레이션
    """
    s3_prefix = DatasetConfig.get_s3_prefix(env, project, version)
    s3_uri = DatasetConfig.get_s3_uri(env, project, version)
    
    print("=" * 60)
    print("데이터셋 S3 업로드")
    print("=" * 60)
    print(f"S3 URI: {s3_uri}")
    print("=" * 60)
    
    uploaded = []
    
    # 데이터셋 파일 업로드
    datasets = {
        'train': train_path,
        'validation': validation_path,
        'test': test_path
    }
    
    print("\n[Step 1] 데이터셋 파일 업로드")
    for name, local_path in datasets.items():
        if local_path and os.path.exists(local_path):
            ext = os.path.splitext(local_path)[1]
            s3_key = f"{s3_prefix}/{name}{ext}"
            uri = upload_file_to_s3(local_path, DatasetConfig.BUCKET, s3_key, dry_run)
            uploaded.append(uri)
        else:
            print(f"⚠ Skipped (not found): {local_path}")
    
    # 메타데이터 생성 및 업로드
    if include_metadata:
        print("\n[Step 2] 메타데이터 업로드")
        with tempfile.TemporaryDirectory() as tmp_dir:
            metadata_path = create_metadata(tmp_dir, metadata_config)
            s3_key = f"{s3_prefix}/metadata.yml"
            uri = upload_file_to_s3(metadata_path, DatasetConfig.BUCKET, s3_key, dry_run)
            uploaded.append(uri)
    
    print("\n" + "=" * 60)
    print(f"완료! S3 Path: {s3_uri}")
    print("=" * 60)
    
    return {'s3_uri': s3_uri, 'uploaded_files': uploaded, 'dry_run': dry_run}

In [26]:
# %% Example Usage
# 방법 1: Placeholder 파일로 테스트 (touch)
print("\n>>> 방법 1: Placeholder 테스트")
result = prepare_and_upload_placeholder(dry_run=True)


>>> 방법 1: Placeholder 테스트
데이터셋 S3 업로드 (Placeholder)
S3 URI: s3://gs-retail-awesome-data-ap-northeast-2/env=dev/project=gs25-sales-forecast/version=v1.0/

[Step 1] Placeholder 파일 생성
✓ Created (placeholder): /tmp/tmp8h41odn7/train.parquet
✓ Created (placeholder): /tmp/tmp8h41odn7/validation.parquet
✓ Created (placeholder): /tmp/tmp8h41odn7/test.parquet
✓ Created: /tmp/tmp8h41odn7/metadata.yml

[Step 2] S3 업로드
[DRY RUN] /tmp/tmp8h41odn7/metadata.yml -> s3://gs-retail-awesome-data-ap-northeast-2/env=dev/project=gs25-sales-forecast/version=v1.0/metadata.yml
[DRY RUN] /tmp/tmp8h41odn7/test.parquet -> s3://gs-retail-awesome-data-ap-northeast-2/env=dev/project=gs25-sales-forecast/version=v1.0/test.parquet
[DRY RUN] /tmp/tmp8h41odn7/validation.parquet -> s3://gs-retail-awesome-data-ap-northeast-2/env=dev/project=gs25-sales-forecast/version=v1.0/validation.parquet
[DRY RUN] /tmp/tmp8h41odn7/train.parquet -> s3://gs-retail-awesome-data-ap-northeast-2/env=dev/project=gs25-sales-forecast/version=v

In [27]:
# %% Example Usage

# 방법 2: 실제 데이터셋 업로드
print("\n>>> 방법 2: 실제 데이터셋 업로드")
result = upload_existing_datasets(
    train_path="./data/train.parquet",
    validation_path="./data/validation.parquet",
    test_path="./data/test.parquet",
    version="v1.1",
    dry_run=True
)


>>> 방법 2: 실제 데이터셋 업로드
데이터셋 S3 업로드
S3 URI: s3://gs-retail-awesome-data-ap-northeast-2/env=dev/project=gs25-sales-forecast/version=v1.1/

[Step 1] 데이터셋 파일 업로드
⚠ Skipped (not found): ./data/train.parquet
⚠ Skipped (not found): ./data/validation.parquet
⚠ Skipped (not found): ./data/test.parquet

[Step 2] 메타데이터 업로드
✓ Created: /tmp/tmp5u2ps8rh/metadata.yml
[DRY RUN] /tmp/tmp5u2ps8rh/metadata.yml -> s3://gs-retail-awesome-data-ap-northeast-2/env=dev/project=gs25-sales-forecast/version=v1.1/metadata.yml

완료! S3 Path: s3://gs-retail-awesome-data-ap-northeast-2/env=dev/project=gs25-sales-forecast/version=v1.1/
